# sukuma-toolkit: train a Sukuma → Swahili translator (Google Colab)

NLLB-200 (Meta) translates Swahili but has no Sukuma. This notebook teaches it Sukuma:

1. **Data**: about 84,000 Sukuma–Swahili Bible verse pairs, split by whole book, plus the UDHR as a non-Bible test.
2. **Baselines**: what you get *without* fine-tuning.
3. **Fine-tune** NLLB-600M (about 30–45 minutes on an L4).
4. **Score** on held-out Bible books and on the UDHR, then try it.

**Copyright:** every Bible used here is copyrighted. The data and the trained model are for research only. They stay in
this session and in a private folder in your Drive; don't commit or publish them.

**Runtime:** Runtime → Change runtime type → **L4** (or A100). Open this notebook from GitHub, not from a Drive copy, so
it always has the latest code.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
REPORTS = '/content/drive/MyDrive/sukuma-toolkit-reports'
MODEL = '/content/drive/MyDrive/sukuma-toolkit-models/nllb-suk-swh'   # private: keep it out of shared folders
!mkdir -p "$REPORTS" /content/drive/MyDrive/sukuma-toolkit-models

In [ ]:
![ -d /content/sukuma-toolkit ] && git -C /content/sukuma-toolkit pull -q || git clone -q https://github.com/hesronkija/sukuma-toolkit.git /content/sukuma-toolkit
%cd /content/sukuma-toolkit
!git log --oneline -1
!pip install -q -e ".[translate]"

## 1. Build the data (a few minutes; downloads the Bibles from Hugging Face)

In [ ]:
!python scripts/build_translation_data.py

## 2. Baselines: copy the input, and NLLB with no Sukuma training (about 5 minutes)

In [ ]:
!python scripts/eval_translator.py --baselines

## 3. Fine-tune (about 30–45 minutes on an L4; keep this tab open)

In [ ]:
!python scripts/train_translator.py --out "$MODEL" --checkpoints /content/checkpoints --batch-size 16 --epochs 1

## 4. Score everything and save the reports

In [ ]:
!python scripts/eval_translator.py --model "$MODEL" --baselines
!cp reports/translation.md reports/translation.json reports/translation_udhr_samples.tsv "$REPORTS"/

## 5. Try it

In [ ]:
%%writefile /tmp/try.txt
Banhu bose bakabyalagwa na wiyabi.
Nalinayega iyene aho nizukwa mhayo gwa kale.
Nalīnayega iyene aho nizūkwa mhayo gwa kale.

In [ ]:
!sukuma translate --model "$MODEL" /tmp/try.txt

## 6. Use it on your Mac

Download the `sukuma-toolkit-models/nllb-suk-swh` folder from Drive (about 2.5 GB) into `models/` in your repo, then:

```bash
pip install -e ".[translate]"
echo "Banhu bose bakabyalagwa na wiyabi." | sukuma translate --model models/nllb-suk-swh
```

`models/` is gitignored, so the model is never committed.